# 실습 4A. 기계독해 (1) — BERT로 **답의 자리를 고른다**

**AI아카데미 [A4021] 언어지능: 언어모델 기반 자연어처리 실습 기초 · 2일차 오전**

지문을 읽고 질문에 답하는 문제입니다.

> **지문**: 한국전자통신연구원(ETRI)은 1976년 설립된 정부출연연구기관으로 대전에 본원을 두고 있다.
> **질문**: ETRI 본원은 어디에 있나?  →  **정답**: `대전`

1일차 세 실습과 결정적으로 다른 점이 있습니다. **정답이 라벨이 아닙니다.**
주제분류는 일곱 개 중 하나, 문장유사도는 0~5 사이의 수, 개체명인식은 토큰마다 태그였습니다.
기계독해의 정답은 **지문 안의 한 구간**입니다. 그래서 모델이 답하는 방식도 달라집니다 —
BERT는 지문의 토큰마다 "여기가 답의 **시작**일 점수"와 "여기가 답의 **끝**일 점수"를 매기고,
가장 점수가 높은 두 자리를 골라 그 사이를 잘라냅니다.

**이 방식의 성질 하나를 꼭 기억해 두세요.** 답을 고르는 것이므로, 이 모델은
**지문에 없는 말을 원리적으로 낼 수 없습니다.** 다음 노트북(실습 4B)에서 T5로 같은 문제를 풀면
그때는 답을 *써냅니다*. 두 방식의 차이가 이 이틀의 핵심입니다.

## 순서와 소요시간

| 절 | 내용 | 시간 |
|---|---|---|
| 1~3 | 환경 확인 · 데이터 보기 · 조각내기 살펴보기 | 10분 |
| 4 | **미션 `m-mrc-1`·`m-mrc-2`·`m-mrc-3`** — 객관식 셋을 풀고 빈칸 세 곳 채우기 (셀 두 개) | 25분 |
| 5~7 | 학습(15초) · 평가 · 맞힌 예·틀린 예 보기 | 10분 |
| 8 | **데이터를 늘리면 얼마나 오르나** — 60,407건으로 다시 학습 | 10분 |
| 9~10 | GPU 비우기 · **내 모델 데모** | 10분 |
| 11~12 | 퀴즈 `q-mrc` · 더 해보기 | 5분 |

> 이 노트북은 `day1/lab_common.py` 를 그대로 씁니다 — 1일차에 쓰던 그 모듈입니다.
> 기계독해에 필요한 함수만 거기에 더해 두었습니다.

## 1. 환경 확인

저장소 **루트**를 작업 폴더로 삼습니다. 공통 모듈은 1일차와 같은 `day1/lab_common.py` 입니다.

In [ ]:
import os, sys, json, time
from pathlib import Path

# 노트북이 day2/ 또는 day2/instructor/ 에 있어도 저장소 루트를 찾아 이동한다.
# 공통 모듈은 1일차에 쓰던 day1/lab_common.py 를 그대로 쓴다.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *_here.parents] if (p / "day1" / "lab_common.py").exists()), None)
assert _root is not None, "DeepKNLP 저장소 안에서 이 노트북을 여세요"
os.chdir(_root); sys.path.insert(0, str(_root / "day1"))
PY = sys.executable                       # 셸 명령(!)에서 이 커널과 같은 파이썬을 쓰기 위해

import lab_common as L
from datasets import Dataset
from transformers import AutoModelForQuestionAnswering
from common import TASKS

TASK = "mrc"
MODEL = L.MODEL_BERT
OUT = L.OUT_DIR_DAY2                      # 결과는 output/day2/ 에 쌓인다
EPOCHS, LR, BATCH = 3, 5e-5, 16          # BERT 학습 설정 (지문이 길어 배치를 줄인다)

L.env_info()

## 2. 데이터 보기 — KorQuAD

학습 **800건**, 평가 **300건**입니다. 1일차와 같은 방식으로, 2일차 오후에 LLM에게 줄 것과
**똑같은 예제들**을 씁니다. 그래야 마지막에 세 방식을 나란히 비교할 수 있습니다.

각 예제는 **지문(context) · 질문(question) · 정답(answer)** 세 부분입니다.
정답에는 글자 위치(`answer_start`)가 함께 붙어 있습니다 — 이것이 미션에서 쓸 재료입니다.

In [ ]:
train_rows = L.load_budget(TASK)
eval_rows = L.load_eval(TASK)
L.check_no_leak(train_rows, eval_rows)

_r = train_rows[0]
print("\n[예제 하나 뜯어보기]")
print("지문 :", _r["input"]["context"][:120], "…")
print("질문 :", _r["input"]["question"])
print("정답 :", _r["raw"]["answer_text"][0], f'(지문의 {_r["raw"]["answer_start"][0]}번째 글자부터)')
print("확인 :", repr(_r["input"]["context"][_r["raw"]["answer_start"][0]:][:len(_r["raw"]["answer_text"][0])]))

In [ ]:
print("[평가셋 예시 3건]")
L.preview(eval_rows, 3)

## 3. 지문이 길면 어떻게 하나 — 조각내기(stride)

1일차에는 문장이 짧아 128토큰이면 넉넉했습니다. 지문은 그렇지 않습니다.
`max_length=384` 를 넘는 지문은 잘리는데, 하필 잘린 뒷부분에 답이 있으면 배울 수가 없습니다.

그래서 **겹쳐 가며 여러 조각으로 나눕니다**. `stride=128` 은 조각끼리 128토큰을 겹치게 한다는 뜻입니다.
겹치기 때문에 답이 조각 경계에 걸려 반토막 나는 일이 줄어듭니다.

결과적으로 **예제 하나가 feature 여러 개**가 됩니다. 그래서 두 가지가 따라옵니다.

- 어떤 조각에는 **답이 아예 없습니다.** 그런 조각은 "답 없음"으로 표시해야 합니다(미션 Step 1).
- 예측할 때는 한 예제의 조각들을 **다시 모아** 가장 점수 높은 답 하나를 골라야 합니다
  (`L.postprocess_mrc` 가 합니다).

In [ ]:
# 그림은 SVG 로 그린다 — 글자로 그린 그림은 한글 폭 때문에 줄이 어긋난다
sys.path.insert(0, str(next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "tools" / "diagram.py").exists()) / "tools"))   # 앞 셀을 건너뛰어도 이 자리에서 다시 찾는다
from diagram import mrc_chunks, show
show(mrc_chunks())

In [ ]:
tokenizer = L.load_bert_tokenizer(MODEL)
_probe = tokenizer([r["input"]["question"].strip() for r in train_rows[:200]],
                   [r["input"]["context"] for r in train_rows[:200]],
                   truncation="only_second", max_length=L.MAX_LEN_MRC, stride=L.STRIDE_MRC,
                   return_overflowing_tokens=True)
print(f"예제 200건 → 조각 {len(_probe['input_ids'])}개 "
      f"(예제당 평균 {len(_probe['input_ids'])/200:.2f}개)")

L.show_tokens(tokenizer, train_rows[0]["input"]["question"], train_rows[0]["input"]["context"][:60])

### 퀴즈 `q-mrc-2` — 조각을 왜 겹쳐서 나누나

방금 본 조각내기에서 **겹침**이 무엇을 막는지 묻는 문제입니다. 보기를 고르면 해설이 열립니다.

In [ ]:
L.quiz("q-mrc-2")

## 4. 미션 `m-mrc` — 답의 시작·끝 토큰 위치 찾기

모델에게 "답은 여기서 시작해서 여기서 끝난다"를 알려 주어야 합니다.
그런데 데이터가 알려주는 것은 **글자 위치**(`answer_start`)이고, 모델이 아는 것은 **토큰 번호**입니다.
그 둘을 잇는 것이 이번 미션입니다. 1일차 `m-ner`(글자 태그 → 서브워드 태그)와 같은 계열의 문제입니다.

재료는 `offset_mapping` 입니다. 토큰마다 `(시작 글자, 끝 글자)` 를 알려줍니다.

```
지문 글자:   … 대  전  에   본  원  을  …
토큰:          [ 대전 ]   [ 에 ]  [ 본원 ]  …
offset:        (31,33)   (33,34)  (35,37)
정답 "대전" → ans_start=31, ans_end=33  →  시작 토큰·끝 토큰을 찾으면 된다
```

세 걸음으로 나뉘는데, **가운데 두 개(`while` 두 줄)는 채워 두었습니다.**
이 셀에서 채울 것은 **`____` 두 곳**입니다. 세 번째 미션(`m-mrc-3`)은 **학습이 끝난 뒤 답을 되돌리는 쪽**이라
확인 셀 다음에 따로 둡니다.

| Step | 판단 | 누가 |
|---|---|---|
| 1 | 이 조각에 답이 통째로 들어 있나? 아니면 무엇을 라벨로? | **`m-mrc-1`** |
| 2 | 답이 시작하는 토큰 찾기 | 채워 두었습니다 |
| 3 | 답이 끝나는 토큰 찾기 | 채워 두었습니다 |
| 4 | `while` 이 한 칸 지나쳐 멈춘 것을 되돌리기 | **`m-mrc-2`** |

아래 객관식 두 개를 먼저 풀어 보세요. 고른 답을 그 다음 셀의 `____` 에 그대로 옮기면 됩니다.
막히면 **힌트**를 단계별로 열 수 있습니다.

In [ ]:
L.quiz("m-mrc-1")

In [ ]:
L.quiz("m-mrc-2")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  미션 m-mrc-1 · m-mrc-2 — 답의 시작·끝 토큰 위치 찾기
# ══════════════════════════════════════════════════════════════════════
# 아래 코드에서 ____ 두 곳만 채우세요. 바로 위 셀의 객관식에서 고른 답을 그대로 옮기면 됩니다.
# 나머지는 다 채워 두었습니다. 채운 뒤 다음 '확인' 셀을 실행하세요.

def build_model():
    # 추출형 QA용 head — 토큰마다 "여기가 답의 시작일 점수"와 "여기가 답의 끝일 점수" 두 개를 낸다.
    # 이번 미션 대상이 아니니 그대로 둡니다.
    return AutoModelForQuestionAnswering.from_pretrained(MODEL)


def encode_mrc_train(tokenizer, rows, max_len=L.MAX_LEN_MRC, stride=L.STRIDE_MRC):
    questions = [r["input"]["question"].strip() for r in rows]
    contexts = [r["input"]["context"] for r in rows]
    # 지문이 길면 stride 만큼 겹쳐 가며 여러 조각으로 나눈다 → 예제 하나가 feature 여러 개가 된다
    enc = tokenizer(questions, contexts, truncation="only_second", max_length=max_len, stride=stride,
                    return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length")
    sample_map = enc.pop("overflow_to_sample_mapping")   # 이 조각이 몇 번째 예제에서 나왔나
    offsets = enc.pop("offset_mapping")                  # 토큰 → (시작 글자, 끝 글자)
    starts, ends = [], []

    for i, off in enumerate(offsets):
        r = rows[sample_map[i]]
        ans_text = r["raw"]["answer_text"][0]
        ans_start = (r["raw"]["answer_start"][0] if r["raw"]["answer_start"]
                     else r["input"]["context"].find(ans_text))
        ans_end = ans_start + len(ans_text)              # 답이 끝난 '다음' 글자 위치
        seq_ids = enc.sequence_ids(i)                    # 0=질문, 1=지문, None=특수토큰
        ctx_s = seq_ids.index(1)                         # 지문이 시작하는 토큰 번호
        ctx_e = len(seq_ids) - 1 - seq_ids[::-1].index(1)  # 지문이 끝나는 토큰 번호

        # Step 1. 이 조각에 답이 통째로 들어 있지 않다 → "답 없음" 표시를 넣는다
        #          위 셀의 객관식 m-mrc-1 에서 고른 값을 ____ 에 옮기세요.
        if not (off[ctx_s][0] <= ans_start and off[ctx_e][1] >= ans_end):
            no_answer = ____                                 # m-mrc-1
            starts.append(no_answer)
            ends.append(no_answer)
            continue

        # Step 2. 지문 왼쪽 끝에서 오른쪽으로 — 시작 글자가 ans_start 를 넘어서기 직전 토큰
        ts = ctx_s
        while ts <= ctx_e and off[ts][0] <= ans_start:
            ts += 1

        # Step 3. 지문 오른쪽 끝에서 왼쪽으로 — 끝 글자가 ans_end 밑으로 내려가기 직전 토큰
        te = ctx_e
        while te >= ctx_s and off[te][1] >= ans_end:
            te -= 1

        # Step 2·3 의 while 은 한 칸 지나쳐 멈춥니다. 한 칸 되돌려 넣습니다.
        #          시작 쪽은 채워 두었습니다. 끝 쪽은 객관식 m-mrc-2 에서 고른 값을 옮기세요.
        starts.append(ts - 1)
        ends.append(____)                                    # m-mrc-2

    enc["start_positions"] = starts
    enc["end_positions"] = ends
    return Dataset.from_dict(dict(enc))

### 확인

라벨이 제대로 만들어졌는지, **라벨이 가리키는 구간을 다시 글자로 풀어 정답과 맞춰** 봅니다.
이 검사를 통과하면 모델에게 옳은 것을 가르치고 있는 것입니다.

> 빈칸을 채우지 않으면 여기서 `NameError: name '____' is not defined` 로 멈춥니다. 정상입니다 —
> 위 셀의 `____` 두 곳을 채우고 그 셀을 다시 실행한 뒤 이 셀을 실행하세요.

In [ ]:
_rows = train_rows[:200]        # Step 1(답 없는 조각)이 실제로 나오려면 예제가 좀 있어야 한다
_ds = encode_mrc_train(tokenizer, _rows)
assert "start_positions" in _ds.column_names, "start_positions 가 없습니다"
assert "end_positions" in _ds.column_names, "end_positions 가 없습니다"
assert len(_ds["start_positions"]) == len(_ds["input_ids"]), "라벨 개수가 조각 개수와 맞지 않습니다"

# 라벨이 가리키는 구간을 다시 글자로 풀어서 정답과 맞춰 본다 — 이게 맞으면 제대로 만든 것이다
_enc = tokenizer([r["input"]["question"].strip() for r in _rows], [r["input"]["context"] for r in _rows],
                 truncation="only_second", max_length=L.MAX_LEN_MRC, stride=L.STRIDE_MRC,
                 return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length")
_smap = _enc["overflow_to_sample_mapping"]

_ok = _checked = 0
for _i, (_s, _e) in enumerate(zip(_ds["start_positions"], _ds["end_positions"])):
    if _s == 0 and _e == 0:
        continue                                   # 이 조각에는 답이 없다고 표시한 것
    _checked += 1
    _row = _rows[_smap[_i]]
    _off = _enc["offset_mapping"][_i]
    _span = _row["input"]["context"][_off[_s][0]: _off[_e][1]].strip()
    _gold = _row["raw"]["answer_text"][0].strip()
    if _span == _gold:
        _ok += 1
    elif _checked <= 3:
        print(f"  라벨이 가리킨 구간 {_span!r} · 정답 {_gold!r}")

_noans = sum(1 for _s, _e in zip(_ds["start_positions"], _ds["end_positions"]) if _s == 0 and _e == 0)

assert _checked > 0, "답이 있다고 표시된 조각이 하나도 없습니다 — Step 1의 판정이 너무 엄격합니다"
assert _noans > 0, ("답이 없는 조각을 하나도 표시하지 않았습니다. 지문을 겹쳐 나눴으니 "
                    "답이 들어 있지 않은 조각이 반드시 생깁니다 — Step 1을 확인하세요")
assert _ok / _checked >= 0.9, f"라벨이 정답 구간을 제대로 가리키지 않습니다 ({_ok}/{_checked})"

print(f"통과 — 예제 {len(_rows)}건 → 조각 {len(_ds)}개 (지문이 길어 조각이 더 많습니다)")
print(f"        답이 든 조각 {_checked}개 중 {_ok}개에서 라벨이 정답 구간을 정확히 가리킵니다")
print(f"        답이 없어 [CLS](0,0)로 표시한 조각 {_noans}개 — Step 1이 한 일입니다")

### 미션 셀 ② — `m-mrc-3` · 시작·끝 점수를 짝지어 답 고르기

위까지가 **모델에 넣는 쪽**(라벨 만들기)이었습니다. 이제 반대쪽입니다 — **모델이 낸 것을 받는 쪽**.

추출형 모델은 답을 한 번에 내지 않습니다. **시작 점수 한 줄, 끝 점수 한 줄**을 따로 냅니다.
둘을 짝지어야 구간이 되고, 조각이 여럿이면 조각을 다 훑어 가장 점수 높은 짝을 골라야 합니다.
아래 함수가 그 일을 합니다. 채울 것은 **`____` 한 곳** — 어떤 짝을 버릴 것인가입니다.

이 함수는 §6 평가에서 300건 전체에 그대로 쓰입니다.

In [ ]:
L.quiz("m-mrc-3")

In [ ]:
# ____ 한 곳만 채우세요. 위 카드에서 고른 답을 그대로 옮기면 됩니다.

def pick_best_span(rows, offsets, sample_map, start_logits, end_logits,
                   n_best=20, max_answer_len=30):
    """조각마다 나온 시작·끝 점수 → 예제마다 답 문자열 하나.

    모델은 **시작 점수 한 줄, 끝 점수 한 줄**을 따로 낸다. 둘을 짝지어야 구간이 된다.
    한 예제가 여러 조각으로 나뉘었으면 조각을 모두 훑어 점수 합이 가장 큰 짝을 고른다.
    말이 안 되는 짝은 버린다 — 지문 밖이거나, 끝이 시작보다 앞이거나, 너무 길거나.
    """
    import numpy as np
    from collections import defaultdict
    by_example = defaultdict(list)
    for i, ex_idx in enumerate(sample_map):
        by_example[ex_idx].append(i)                 # 예제 → 그 예제의 조각들

    preds = []
    for ex_idx, r in enumerate(rows):
        context = r["input"]["context"]
        best, best_score = "", -1e9
        for fi in by_example[ex_idx]:
            off = offsets[fi]
            s_idx = np.argsort(start_logits[fi])[-1: -n_best - 1: -1]    # 시작 점수 상위 n_best
            e_idx = np.argsort(end_logits[fi])[-1: -n_best - 1: -1]      # 끝 점수 상위 n_best
            for s in s_idx:
                for e in e_idx:
                    # 말이 안 되는 짝은 점수가 높아도 버린다
                    if off[s] is None or off[e] is None or ____ or e - s + 1 > max_answer_len:   # m-mrc-3
                        continue
                    score = start_logits[fi][s] + end_logits[fi][e]
                    if score > best_score:
                        best_score = score
                        best = context[off[s][0]: off[e][1]]
        preds.append(best.strip())
    return preds

### 확인

가짜 점수를 넣어 **뒤집힌 짝을 버리는지**만 봅니다. 모델을 돌리지 않으므로 바로 끝납니다.

In [ ]:
import numpy as _np
# 지문 "대전에 본원을" — 토큰 3개: [대전][에][본원을]. 정답은 "대전".
_rows = [{"input": {"context": "대전에 본원을"}}]
_off = [[(0, 2), (2, 3), (4, 6)]]                      # 토큰마다 (시작 글자, 끝 글자)
_map = [0]                                             # 조각 0 → 예제 0
# 함정을 심는다: 시작=토큰2, 끝=토큰0 인 **뒤집힌 짝**의 점수 합(6+6=12)이 가장 크다.
# 말이 되는 짝 중 최고는 (0,0) = 4+6 = 10 → "대전".
_start = [_np.array([4.0, 0.0, 6.0])]
_end = [_np.array([6.0, 0.0, 3.0])]
_got = pick_best_span(_rows, _off, _map, _start, _end)
assert _got == ["대전"], f"기대: ['대전'] / 받은 값: {_got}  ← 뒤집힌 짝이 걸러졌는지 보세요"
print("통과 — 점수가 가장 높은 뒤집힌 짝을 버리고 '대전' 을 골랐습니다")

## 5. 학습

설정은 `task5-llm-ft/bert_baseline.py --task mrc --mode budget` 과 같습니다 —
epochs 3, 학습률 5e-5, 최대 길이 384. 배치만 16으로 줄였습니다(지문이 길어 한 칸이 큽니다).
강의장 GPU에서 **20초 안팎**이면 끝납니다.

In [ ]:
L.set_seed(42)

train_ds = encode_mrc_train(tokenizer, train_rows)
eval_ds, eval_offsets, eval_map = L.encode_mrc_eval(tokenizer, eval_rows)
model = build_model()
print(f"학습 조각 {len(train_ds)}개 (예제 {len(train_rows)}건) / 평가 조각 {len(eval_ds)}개 (예제 {len(eval_rows)}건)")

trainer, bert_meta = L.train_bert(model, tokenizer, train_ds, TASK, epochs=EPOCHS, lr=LR,
                                  batch_size=BATCH, max_len=L.MAX_LEN_MRC)

# 학습한 모델을 디스크에 남긴다 — 뒤에서 데모 서버가 이것을 읽는다 (약 440MB)
L.save_model(model, tokenizer, "mrc", "bert")

## 6. 평가

조각마다 나온 시작·끝 점수를 **예제 단위로 다시 모아** 가장 점수가 높은 구간 하나를 고릅니다
— 미션 셀 ②에서 만든 `pick_best_span` 이 그 일을 합니다. 그렇게 얻은 답 문자열을 LLM·T5와 **똑같은 채점 함수**에 넣습니다.

지표가 둘입니다.

- **EM(exact match)** — 답이 정답과 글자까지 똑같은 비율. 엄격합니다.
- **F1** — 글자 단위로 겹치는 정도. "대전광역시"와 "대전"처럼 **부분적으로 맞은 답**에 점수를 줍니다.

In [ ]:
# 미션 셀 ②에서 만든 pick_best_span 이 여기서 300건 전체에 쓰입니다
preds = L.predict_bert(TASK, trainer, eval_ds, tokenizer, eval_rows,
                       offsets=eval_offsets, sample_map=eval_map, postprocess=pick_best_span)
print("예측 예시 3건:", preds[:3], "\n")

bert_summary = L.evaluate(TASK, preds, eval_rows)
L.save_result(TASK, "bert", bert_summary, bert_meta, out_dir=OUT)

### 퀴즈 `q-mrc-3` — 조각마다 답을 내지 않고 예제 단위로 모으는 이유

방금 평가에서 조각마다 답을 내지 않고 **예제 단위로 모은** 이유를 묻는 문제입니다. 보기를 고르면 해설이 열립니다.

In [ ]:
L.quiz("q-mrc-3")

## 7. 맞힌 예와 틀린 예 보기

**잘 맞힌 세 건을 먼저, 크게 틀린 세 건을 그다음에** 봅니다.
EM 76%란 넷 중 셋을 글자까지 똑같이 맞혔다는 뜻입니다 — 맞힌 쪽을 먼저 보셔야 그 감이 잡힙니다.

그다음 틀린 쪽에서 **어떻게** 틀렸는지 보세요.
답의 자리를 아주 엉뚱하게 짚었나요, 아니면 경계만 조금 넓거나 좁게 잡았나요?
후자라면 EM은 0점이지만 F1은 높게 나옵니다 — `채점` 칸의 점수가 그것입니다.

In [ ]:
L.examples(TASK, preds, eval_rows, 3)      # 맞힌 것 3건 + 틀린 것 3건

## 8. 데이터를 늘리면 얼마나 오르나 — 같은 모델, 데이터만 수십 배

지금까지 쓴 학습 예제는 **800건**입니다. 왜 그것뿐이었냐면, 오후 실습5의 LLM 이 배우는 양과
**똑같이 맞춰야** BERT·T5·GPT 계열을 나란히 놓고 비교할 수 있기 때문입니다.
그래서 학습이 15초 만에 끝났습니다 — 무엇이 일어나는지 볼 겨를도 없이.

이번에는 **같은 모델을 KorQuAD 공식 train 으로** 학습해 봅니다 — 전체 파일(60,407건)이 있으면 그것으로, 이 저장소처럼 없으면 절반 세트(30,000건)로 자동 대신합니다. 화면의 학습 예제 수를 확인하세요.
75배 많은 데이터입니다. 몇 분 걸리므로, 그동안 검증 점수가 어떻게 올라가는지 지켜보세요.

> ⚠️ **여기서 나오는 점수는 비교 표에 넣지 않습니다.** 데이터가 다르면 비교가 성립하지
> 않습니다. §7의 표와 오후 실습5의 비교는 **언제나 800건 쪽 값**으로 합니다.
> 이 절은 "같은 모델·같은 코드에서 데이터만 늘리면 어떻게 되나" 하나만 봅니다.

`L.load_full` 은 공식 train 전체를 읽되 **평가셋과 겹치는 예제는 빼고** 줍니다.
그래야 점수가 부풀지 않습니다. 터미널에서 같은 것을 하려면
`python task5-llm-ft/bert_baseline.py --task mrc --mode full` 입니다.

In [ ]:
# 시간이 없으면 FULL_LIMIT 를 20000 쯤으로 줄이세요 (그만큼 빨리 끝나고 점수는 조금 낮습니다)
FULL_LIMIT = None                       # None = 파일에 든 전부 (전체 60,407건 또는 절반 30,000건)

full_rows = L.load_full(TASK, limit=FULL_LIMIT)
full_ds = encode_mrc_train(tokenizer, full_rows)      # §4에서 여러분이 채운 그 함수입니다
print(f"학습 예제 {len(full_rows):,}건 → 조각 {len(full_ds):,}개")

In [ ]:
# 같은 코드, 같은 모델. 데이터만 다릅니다. 1 epoch 만 돕니다 (예제가 많아 그것으로 충분합니다)
L.set_seed(42)
full_model = build_model()
full_trainer, full_meta = L.train_bert(full_model, tokenizer, full_ds, TASK, epochs=1.0)

In [ ]:
# 같은 평가셋, 같은 채점 함수로 잽니다
full_ds_eval, full_offsets, full_map = L.encode_mrc_eval(tokenizer, eval_rows)
full_preds = L.predict_mrc(full_trainer, full_ds_eval, eval_rows, full_offsets, full_map)
full_summary = L.evaluate(TASK, full_preds, eval_rows)

import pandas as pd
print()
display(pd.DataFrame([
    {"학습 예제": f"{len(train_rows):,}건 (실습5와 같은 양)", "EM": bert_summary["em"],
     "F1": bert_summary["f1"], "학습(분)": round(bert_meta["train_seconds"] / 60, 1)},
    {"학습 예제": f"{len(full_rows):,}건 (공식 train 전체)", "EM": full_summary["em"],
     "F1": full_summary["f1"], "학습(분)": round(full_meta["train_seconds"] / 60, 1)},
]).set_index("학습 예제"))

> **관찰 포인트.** 데이터를 75배 늘리면 점수가 몇 배 오르나요? 그렇지 않을 것입니다 —
> **처음 800건이 가장 크게 기여하고, 그다음부터는 조금씩 오릅니다.** 학습 시간은 정직하게 75배 가까이 늘어납니다.
>
> 이것이 오후 실습5의 배경이기도 합니다. LLM 은 800건만 보고도 여섯 태스크를 하는데,
> 그것은 **이미 배운 것이 많아서**입니다. 데이터를 더 넣는 것과 더 큰 모델을 쓰는 것은 다른 선택지입니다.

## 9. GPU 비우기 — 다음 노트북을 위해

다음 노트북(실습 4B)의 T5는 **이 모델보다 GPU를 훨씬 많이 씁니다**(지문이 길어 약 11GB).
이 노트북의 모델을 메모리에서 내리고 넘어가세요. 안 그러면 다음 노트북에서
`CUDA out of memory` 가 날 수 있습니다.

In [ ]:
# 나중에 다시 돌아와 이 셀만 실행해도 되게 — model·trainer 가 없으면 조용히 넘어간다
for _n in ("model", "trainer"):
    globals().pop(_n, None)
L.free_gpu()
import torch
if torch.cuda.is_available():
    print(f"지금 GPU 사용량: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print("실습 4B(05_기계독해_T5.ipynb)로 넘어가세요.")

## 10. 내 모델 데모 — 방금 학습시킨 BERT에게 직접 물어보기

점수표는 300건의 평균입니다. **내가 넣은 문장 하나**에 이 모델이 어떻게 답하는지는 직접 넣어 봐야 압니다.
`day2/serve_mrc.py` 가 작은 웹 페이지를 띄웁니다 — 지문과 질문을 넣으면 방금 학습한 모델이 답을 냅니다.

**아래 켜기 셀로 띄웁니다.** 서버는 배경에서 돌고 셀은 바로 끝나므로 노트북을 계속 진행할 수 있습니다.
다 보고 나면 **끄기 셀**로 내립니다 — 필요할 때 몇 번이든 켰다 껐다 할 수 있습니다. 터미널에서 띄워도 됩니다:

```bash
python day2/serve_mrc.py --kind bert --port 9006
```

브라우저에서 <http://localhost:9006> 을 엽니다. 다른 자리에서 접속하려면 `localhost` 대신 그 PC의 IP를 씁니다.
노트북에서 켰다면 아래 **끄기 셀**로 끕니다. 터미널에서 직접 띄웠다면 `Ctrl+C` 입니다.

> 위 9절에서 GPU를 이미 비웠습니다. 데모 서버는 저장된 모델을 **새로 읽어** 올리므로 그 편이 안전합니다.

In [ ]:
# ▶ 켜기 — 배경에서 서버를 띄우므로 이 셀은 곧 끝나고 노트북을 계속 쓸 수 있습니다
L.demo_start("bert", port=9006)

In [ ]:
# ■ 끄기 — 다 보고 나면 실행하세요. 다시 보려면 위 켜기 셀을 다시 실행하면 됩니다
L.demo_stop(9006)

**넣어 볼 것들** — 학습 데이터에 없는 종류를 골라야 차이가 보입니다.

- **답이 지문에 없는 질문.** 지문에 없는 것을 물어보세요. 추출형은 **지어낼 수 없습니다** —
  지문 안의 위치만 고르기 때문입니다. 무엇을 고르는지 보세요.
- **지문에 없는 표현으로 묻기.** 지문이 "설립되었다" 인데 "언제 만들어졌나요?" 로 물어보세요.
  단어가 겹치지 않아도 찾아내는지.
- **숫자·날짜.** 지문에 숫자가 여럿 있을 때 맞는 것을 고르는지.
- **지문을 직접 붙여 넣기.** 오늘 아침 뉴스 한 문단을 넣고 물어보세요. KorQuAD(위키백과)와
  글의 결이 다를 때 어떻게 되는지.

> **관찰 포인트.** 모델이 틀릴 때 **엉뚱한 위치를 고르는지**, 아니면 **답이 없다고 하는지** 보세요.
> 그 둘은 다른 실패입니다. 내일 실습 4B의 T5는 여기서 또 다르게 실패합니다.

## 11. 퀴즈 `q-mrc`

오늘 본 두 방식의 차이를 한 문장으로 정리해 보는 문제입니다. 보기를 고르면 해설이 열립니다.

In [ ]:
L.quiz("q-mrc")

## 12. 더 해보기

1. **`stride` 를 0으로 두면?** 조각이 겹치지 않게 됩니다. 답이 조각 경계에 걸리는 예제가 생기고,
   점수가 어떻게 되는지 보세요.
2. **`max_answer_len`(기본 30)을 5로 줄이면?** 미션 셀 ②의 `pick_best_span` 인자입니다.
   긴 정답을 아예 못 고르게 되면 EM과 F1이 어떻게 갈리는지 보세요.
3. **데이터를 늘리면?** 터미널에서 (공식 train 전체):

   ```
   python task5-llm-ft/bert_baseline.py --task mrc --mode full --save output/day2/mrc-bert-full.json
   ```

4. **답이 없는 질문을 주면?** 지문과 상관없는 질문을 만들어 넣어 보세요.
   추출형 모델은 "모르겠다"고 답할 수 없습니다 — 무엇을 내놓나요?